In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pickle

import tensorflow
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

In [9]:

df = pd.read_csv('CreditCardData.csv')

df.drop(["Transaction ID"], axis=1, inplace=True)
df.dropna(axis=0, inplace=True)
df["Amount"] = df["Amount"].str.replace("£", "").astype(float)
df["Date"] = pd.to_datetime(df["Date"])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df.drop(["Date"], axis=1, inplace=True)
df.drop(["Month", "Year"], axis=1, inplace=True)


X = df.drop(columns=['Fraud'])
y = df['Fraud']


categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(exclude=['object']).columns.tolist()


numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)




X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)


X_train = X_train.toarray()
X_test = X_test.toarray()



X_train_reshaped = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_reshaped = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}


model = Sequential()
model.add(LSTM(128, input_shape=(X_train_reshaped.shape[1], X_train_reshaped.shape[2]), return_sequences=True))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(LSTM(64, return_sequences=True))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(LSTM(32))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(1, activation='sigmoid'))


model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


model.fit(X_train_reshaped, y_train, epochs=10, batch_size=32, validation_split=0.2)


/tmp/ipykernel_4296/2968781893.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"])
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.9378 - loss: 0.1657 - val_accuracy: 0.9774 - val_loss: 0.0578
Epoch 2/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.9703 - loss: 0.0758 - val_accuracy: 0.9803 - val_loss: 0.0522
Epoch 3/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.9748 - loss: 0.0658 - val_accuracy: 0.9809 - val_loss: 0.0488
Epoch 4/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9776 - loss: 0.0600 - val_accuracy: 0.9819 - val_loss: 0.0500
Epoch 5/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9798 - loss: 0.0566 - val_accuracy: 0.9837 - val_loss: 0.0459
Epoch 6/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9804 - loss: 0.0533 - val_accuracy: 0.9848 - val_loss: 0.0418
Epoch 7/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.9815 - loss: 0.0529 - val_accuracy: 0.9844 - val_loss: 0.0436
Epoch 8/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - accuracy: 0.9823 - loss: 0

In [11]:

import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Check unique classes
unique_classes = np.unique(y_train)
print("Unique classes in y_train:", unique_classes)

if len(unique_classes) == 2:

    # Map classes to 0 and 1
    class_mapping = {
        unique_classes[0]: 0,
        unique_classes[1]: 1
    }

    y_train_mapped = np.array(
        [class_mapping[i] for i in y_train],
        dtype=np.int32
    )

    y_test_mapped = np.array(
        [class_mapping[i] for i in y_test],
        dtype=np.int32
    )


    # Ensure X is a proper numpy array
    X_train_reshaped = np.asarray(X_train_reshaped, dtype=np.float32)


    # Check shapes
    print("X_train shape:", X_train_reshaped.shape)
    print("y_train shape:", y_train_mapped.shape)


    # Compute class weights
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train_mapped),
        y=y_train_mapped
    )

    class_weight_dict = {
        0: class_weights[0],
        1: class_weights[1]
    }

    print("Class weights:", class_weight_dict)


    # Train model
    history = model.fit(
        X_train_reshaped,
        y_train_mapped,
        epochs=20,
        batch_size=64,
        validation_split=0.2,
        verbose=2,
        class_weight=class_weight_dict
    )

else:
    print("Unexpected number of classes. Check your labels.")

Unique classes in y_train: [0 1]
X_train shape: (79981, 1, 51)
y_train shape: (79981,)
Class weights: {0: np.float64(0.5385925925925926), 1: np.float64(6.977927063339731)}
Epoch 1/20
1000/1000 - 12s - 12ms/step - accuracy: 0.9384 - loss: 0.0958 - val_accuracy: 0.9341 - val_loss: 0.1018
Epoch 2/20
1000/1000 - 10s - 10ms/step - accuracy: 0.9374 - loss: 0.0953 - val_accuracy: 0.9400 - val_loss: 0.0816
Epoch 3/20
1000/1000 - 10s - 10ms/step - accuracy: 0.9394 - loss: 0.0939 - val_accuracy: 0.9382 - val_loss: 0.0909
Epoch 4/20
1000/1000 - 9s - 9ms/step - accuracy: 0.9390 - loss: 0.0930 - val_accuracy: 0.9400 - val_loss: 0.0808
Epoch 5/20
1000/1000 - 10s - 10ms/step - accuracy: 0.9382 - loss: 0.0904 - val_accuracy: 0.9451 - val_loss: 0.0785
Epoch 6/20
1000/1000 - 10s - 10ms/step - accuracy: 0.9413 - loss: 0.0884 - val_accuracy: 0.9351 - val_loss: 0.0930
Epoch 7/20
1000/1000 - 11s - 11ms/step - accuracy: 0.9408 - loss: 0.0881 - val_accuracy: 0.9368 - val_loss: 0.0957
Epoch 8/20
1000/1000 - 8s

In [24]:
with open('credit_card_fraud_preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

model.save('credit_card_fraud_model_lstm.keras')

In [25]:
import pickle

# Save preprocessing pipeline
with open('credit_card_fraud_preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

# Save model in the recommended Keras format
model.save('credit_card_fraud_model_lstm.keras')

print("Preprocessor and model saved successfully!")

Preprocessor and model saved successfully!


In [26]:
from keras.models import load_model

model = load_model('credit_card_fraud_model_lstm.keras')

print("Model loaded successfully!")

Model loaded successfully!


In [27]:
import os

print("Current working directory:", os.getcwd())
print("\nFiles in current directory:")
print(os.listdir())

Current working directory: /content

Files in current directory:
['.config', 'CreditCardData.csv', 'credit_card_fraud_model_lstm.keras', 'credit_card_fraud_model_lstm.h5', 'credit_card_fraud_preprocessor.pkl', 'sample_data']


In [18]:
from google.colab import files

files.download("credit_card_fraud_preprocessor.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>